movieplotsbow utilizes bag of words to give movie recommendations based off of word frequency. If you want to watch a movie about dinosaurs, you'd put dinosaurs in. If you want to watch a murder mystery, put in murder. The idea is that it will give you 5 plots that frequently use the user word. The original dataset was too large to even upload to github, so for the purpose of getting the dataset to submit, I had to cut it in half. For speed and testing purposes, the dataframe utilized is also set to being 10% of the whole dataset, but can be easily adjusted by removing the df_subset variable and replacing all uses of it with df.

In [3]:
#Import all necessary tools
import pandas as pd
import spacy
from sklearn.feature_extraction.text import CountVectorizer
import random
from sklearn.metrics.pairwise import cosine_similarity

nlp = spacy.load("en_core_web_sm")

#Preprocess the text: Tokenize the data and remove stopwords
def preprocess(text):
    doc = nlp(text)
    tokens = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and token.is_alpha
    ]
    return " ".join(tokens)

#Find the plots that utilize that word and return the top 5
def generate_plot_from_word(vectorizer, bow_matrix, word, top_n=5, df_subset=None):
    cleaned_word = preprocess(word)
    word_vec = vectorizer.transform([cleaned_word])
    similarities = cosine_similarity(word_vec, bow_matrix).flatten()
    top_indices = similarities.argsort()[::-1][:top_n]

    results = []
    for i in top_indices:
        title = df_subset.iloc[i]['Title']
        plot = df_subset.iloc[i]['Plot']
        results.append((title, plot))
    return results


In [4]:
# Load the CSV file
df = pd.read_csv('/content/half_movies.csv')  # Replace with your actual file path
df_subset = df.sample(frac=0.1, random_state=42).dropna(subset=['Plot'])  # Sample 10% for speed
print("Subset of plots created")

#Preprocess plots
cleaned_plots = [preprocess(plot) for plot in df_subset['Plot']]
print("Plots preprocessed")

#Bag of Words Vectorization
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(cleaned_plots)
print("BoW Vectorization Complete")

#Look at Vocabulary and Shape
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("BoW matrix shape:", bow_matrix.shape)

#Word frequency analysis
word_counts = bow_matrix.toarray().sum(axis=0)
vocab = vectorizer.get_feature_names_out()
word_freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
top_words = word_freq_df.sort_values(by='count', ascending=False)
print(top_words.head(20))

#Generate plots based on user word
user_word = "magic"
generated_results = generate_plot_from_word(vectorizer, bow_matrix, user_word, top_n=5, df_subset=df_subset)

for i, (title, plot) in enumerate(generated_results, start=1):
   print(f"\n{user_word} - Top Plot {i} — Title: {title}\nPlot:\n{plot}\n{'='*60}")


Subset of plots created
Plots preprocessed
BoW Vectorization Complete
Vocabulary size: 25506
BoW matrix shape: (1745, 25506)
         word  count
8247     find   2087
22616    tell   1885
12461    kill   1819
13027   leave   1754
13925     man   1603
22389    take   1388
23425     try   1268
19094  return   1262
9385       go   1214
15694     new   1094
10670    home   1072
8014   father   1031
10366    help    989
22949    time    954
12910   later    952
13547    love    949
8797   friend    944
5504      day    942
14532    meet    929
13213    life    908

magic - Top Plot 1 — Title: Telling Lies in America
Plot:
Karchy Jonas (Brad Renfro) is a 17-year-old high-school student (who emigrated from Hungary 7 years earlier) trying to find his way in the world. He meets radio personality Billy Magic (Kevin Bacon) who takes him under his wing. However, authorities are after Billy for accepting payola from record companies to give their songs air time. Billy picks Karchy as when he figure

Find movies with the word Princess

In [5]:
user_word = "princess"
generated_results = generate_plot_from_word(vectorizer, bow_matrix, user_word, top_n=5, df_subset=df_subset)

for i, (title, plot) in enumerate(generated_results, start=1):
   print(f"\n{user_word} - Top Plot {i} — Title: {title}\nPlot:\n{plot}\n{'='*60}")


princess - Top Plot 1 — Title: The Smiling Lieutenant
Plot:
In Vienna, Lieutenant Nikolaus "Niki" von Preyn (Maurice Chevalier) meets Franzi (Claudette Colbert), the leader of an all-female-orchestra. They soon fall in love with each other. While standing in formation before a parade honoring the visiting royal family of Flausenthurm, Niki takes the opportunity to wink at Franzi in the crowd. Unfortunately the gesture is intercepted by Anna, the Princess of Flausenthurm (Miriam Hopkins). The naive Princess assumes offense, leading the lieutenant to convince her that he slighted her because she is thought to be very beautiful. Besotted, the Princess demands she has to marry the lieutenant, or, she'll marry an American instead. The international incident is narrowly averted by having them get married.
The Lieutenant sneaks away from his bride to wander the streets of Flausenthurm to find his girlfriend. The princess learns of this and decides to confront Franzi. After the initial confro

What happens if we add those two search words as stop words?

In [6]:
custom_stop_words = ["magic", "princess"]
for word in custom_stop_words:
    nlp.vocab[word].is_stop = True

# Preprocess plots
cleaned_plots = [preprocess(plot) for plot in df_subset['Plot']]
print("Plots preprocessed")

# Bag of Words Vectorization
vectorizer = CountVectorizer()
bow_matrix = vectorizer.fit_transform(cleaned_plots)
print("BoW Vectorization Complete")

# Optional: Inspect Vocabulary and Shape
print("Vocabulary size:", len(vectorizer.vocabulary_))
print("BoW matrix shape:", bow_matrix.shape)

# Word frequency analysis
word_counts = bow_matrix.toarray().sum(axis=0)
vocab = vectorizer.get_feature_names_out()
word_freq_df = pd.DataFrame({'word': vocab, 'count': word_counts})
top_words = word_freq_df.sort_values(by='count', ascending=False)
print(top_words.head(20))

Plots preprocessed
BoW Vectorization Complete
Vocabulary size: 25506
BoW matrix shape: (1745, 25506)
         word  count
8247     find   2087
22616    tell   1885
12461    kill   1819
13027   leave   1754
13925     man   1603
22389    take   1388
23425     try   1268
19094  return   1262
9385       go   1214
15694     new   1094
10670    home   1072
8014   father   1031
10366    help    989
22949    time    954
12910   later    952
13547    love    949
8797   friend    944
5504      day    942
14532    meet    929
13213    life    908


**Will there be any results matched to the stop word "magic"?**

In [7]:
user_word = "magic"
generated_results = generate_plot_from_word(vectorizer, bow_matrix, user_word, top_n=5, df_subset=df_subset)

for i, (title, plot) in enumerate(generated_results, start=1):
   print(f"\n{user_word} - Top Plot {i} — Title: {title}\nPlot:\n{plot}\n{'='*60}")


magic - Top Plot 1 — Title:  Camp
Plot:
The film centers on the experiences of several teenagers at Camp Ovation, a summer theatre camp. The campers are a mishmash of youngsters from different backgrounds, dealing with different issues, but all feel most comfortable while performing. School is out, and the campers reunite as they prepare to board the buses to Camp Ovation. Shy, self-conscious Ellen Lucas happily greets her friends, recovering from an embarrassing situation where she desperately tried to land a prom date, and ended up asking her brother. Nerdy Fritzi Wagner attempts to befriend icy diva Jill Simmons, but ends up toting her luggage instead. Returning camper Jenna Malloran laments that her parents forced her to have her jaw wired shut in order to lose weight. Upon arrival to Camp Ovation, the kids settle in. Ellen converses with her friend Michael Flores, a gay teenager who was violently beaten by his classmates after showing up to his junior prom in drag. The campers ar

**Will there be any results for the stop word "princess"?**

In [8]:
user_word = "princess"
generated_results = generate_plot_from_word(vectorizer, bow_matrix, user_word, top_n=5, df_subset=df_subset)

for i, (title, plot) in enumerate(generated_results, start=1):
   print(f"\n{user_word} - Top Plot {i} — Title: {title}\nPlot:\n{plot}\n{'='*60}")


princess - Top Plot 1 — Title:  Camp
Plot:
The film centers on the experiences of several teenagers at Camp Ovation, a summer theatre camp. The campers are a mishmash of youngsters from different backgrounds, dealing with different issues, but all feel most comfortable while performing. School is out, and the campers reunite as they prepare to board the buses to Camp Ovation. Shy, self-conscious Ellen Lucas happily greets her friends, recovering from an embarrassing situation where she desperately tried to land a prom date, and ended up asking her brother. Nerdy Fritzi Wagner attempts to befriend icy diva Jill Simmons, but ends up toting her luggage instead. Returning camper Jenna Malloran laments that her parents forced her to have her jaw wired shut in order to lose weight. Upon arrival to Camp Ovation, the kids settle in. Ellen converses with her friend Michael Flores, a gay teenager who was violently beaten by his classmates after showing up to his junior prom in drag. The campers

**Wow, matching on a stop word returns the same results.   That would indicate there are no good matches for the stop word and all results are weighted evenly.**